In [1]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display
import IPython.display as ipd
import sklearn
from sklearn.svm import SVC
import sys
from glob import glob
from itertools import cycle
import seaborn as sns
from pathlib import Path
from collections import Counter
from sklearn.preprocessing import LabelEncoder


sns.set_theme(style="white", palette=None)
color_pal = plt.rcParams["axes.prop_cycle"].by_key()["color"]
color_cycle = cycle(plt.rcParams["axes.prop_cycle"].by_key()["color"])

Preprocessing Function

In [2]:
def cleanUp(audioInfo):
   X = [item["features"] for item in audioInfo]
   Y = [item["chord"] for item in audioInfo]
   X = np.array(X)
   Y = np.array(Y)
   # print(X.shape)
   # print(Y.shape)
   # print(Counter(Y))
   return X, Y


In [3]:
def create_dataset(audioFiles):
    audioInfo = []
    target_duration = 5
    for file in audioFiles:
        y, sr = librosa.load(file)
        if(len(y) > target_duration * sr):
            y_fixed = y[:target_duration * sr]
        else:
            padding = (target_duration * sr) - len(y)
            y_fixed = np.pad(y, (0, padding), mode="constant")
        S = librosa.feature.melspectrogram(y=y_fixed, sr=sr, n_mels=128)
        S_db_mel = librosa.amplitude_to_db(S, ref=np.max)
        chordName = Path(file).parent.name
        audioInfo.append({"features": S_db_mel, "chord": chordName})
    return cleanUp(audioInfo)

In [4]:
trainFiles = glob("./Training/*/*.wav")
testFiles = glob("./Test/*/*.wav")
X_train, Y_train = create_dataset(trainFiles)
X_test, Y_test = create_dataset(testFiles)

In [5]:
encoder = LabelEncoder()
y_train_encoded = encoder.fit_transform(Y_train)
y_test_encoded = encoder.transform(Y_test)

X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_test_flat = X_test.reshape(X_test.shape[0], -1)

# print(X_train_flat.shape)
# print(X_test_flat.shape)


# print(encoder.classes_)
# print(y_train_encoded[350:1000])
# print(y_test_encoded[350:1000])

Training the First Model:

In [6]:
model = SVC(kernel = "rbf")
model.fit(X_train_flat, y_train_encoded)
accuracy = model.score(X_test_flat, y_test_encoded)
print(f"Accuracy: {accuracy:.2f}")

Accuracy: 0.92


In [7]:
# import joblib
# joblib.dump(model, "chord_model.pkl")
# joblib.dump(encoder, "chord_encoder.pkl")

Test with my own guitar

In [8]:
def testAudio(file):
    target_duaration = 5
    y,sr = librosa.load(file)
    target_samples = target_duaration * sr
    if len(y) > target_samples:
        y_fixed = y[:target_samples]
    else:
        padding = target_samples - len(y)
        y_fixed = np.pad(y, (0, padding), mode="constant")
    S = librosa.feature.melspectrogram(y=y_fixed, sr=sr, n_mels=128)
    S_db_mel = librosa.amplitude_to_db(S, ref=np.max)
    features = S_db_mel.reshape(1, -1)
    prediction = model.predict(features)
    chord = encoder.inverse_transform(prediction)[0]
    return chord

In [20]:
myPlays = glob("./MyPlays/*.wav")
# print(myPlays)
print(f"Chord Played: {testAudio(myPlays[0])}") #C
print(f"Chord Played: {testAudio(myPlays[1])}") #Em
print(f"Chord Played: {testAudio(myPlays[2])}") #G
print(f"Chord Played: {testAudio(myPlays[3])}") #Em

Chord Played: C
Chord Played: Em
Chord Played: G
Chord Played: Bb


In [21]:
y, sr = librosa.load("Training/Em/Em_acousticguitar_Mari_1.wav")  # any real training file
onset_frames = librosa.onset.onset_detect(y=y, sr=sr, units="time")
print("Onset(s) at:", onset_frames, "out of total duration:", len(y)/sr)

Onset(s) at: [0.06965986 0.39473923 0.7662585  0.92879819 1.09133787 1.2306576
 1.36997732 1.41641723 1.4860771  1.53251701 2.39165533] out of total duration: 3.1699773242630385


In [ ]:
y, sr = librosa.load("MyPlays/testE_Minor.wav")  # any real testing file
onset_frames = librosa.onset.onset_detect(y=y, sr=sr, units="time")
print("Onset(s) at:", onset_frames, "out of total duration:", len(y)/sr)

Onset(s) at: [0.34829932 4.36535147] out of total duration: 5.0
